In [ ]:
# ================= IMPORT LIBRARIES =================
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from textblob import TextBlob
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from xgboost import XGBRegressor

plt.style.use('seaborn')


# ================= SENTIMENT =================
def fetch_sentiment_score(ticker):
    try:
        ticker_data = yf.Ticker(ticker)
        summary = ticker_data.info.get("longBusinessSummary", "")
        sentiment = TextBlob(summary).sentiment.polarity
        return round(sentiment * 100, 2)
    except:
        return 0.0


# ================= FETCH STOCK DATA =================
def fetch_stock_data(stock, chart_range):
    try:
        ticker = yf.Ticker(stock)
        info = ticker.info
        hist = ticker.history(period=chart_range)
        fin = ticker.financials
        bs = ticker.balance_sheet
        if hist.empty:
            return None
        return {"info": info, "hist": hist, "financials": fin, "balance_sheet": bs}
    except Exception:
        return None


# ================= SCORING FUNCTION =================
def compute_score_grouped_fixed(info, fin, bs, hist, own_stock=False, horizon=None, avg_price=None, quantity=None):
    reasons = []
    group_scores = {}

    def clamp_score(value):
        return max(0, min(100, round(value, 1)))

    # ========= GROUP 1: Valuation =========
    val_score = 50
    pe = info.get('trailingPE', None)
    pb = info.get('priceToBook', None)

    if pe:
        if pe < 15: val_score += 25; reasons.append("Very low P/E → undervalued.")
        elif pe < 30: val_score += 10; reasons.append("Moderate P/E ratio.")
        elif pe > 50: val_score -= 20; reasons.append("Very high P/E → overvalued.")
    else:
        reasons.append("P/E ratio not available.")

    if pb and pb < 3: 
        val_score += 10; reasons.append("Healthy P/B ratio (<3).")
    else:
        reasons.append("P/B ratio is high (>3).")

    group_scores["Valuation"] = clamp_score(val_score)

    # ========= GROUP 2: Profitability =========
    profit_score = 50
    profit_margin = info.get('profitMargins', 0)
    if profit_margin > 0.15: profit_score += 20; reasons.append("High profit margin (>15%).")
    elif profit_margin < 0.05: profit_score -= 20; reasons.append("Low profit margin (<5%).")

    net_income = fin.get('Net Income', pd.Series(dtype=float)).dropna()
    equity = bs.get('Total Stockholder Equity', pd.Series(dtype=float)).dropna()
    if not net_income.empty and not equity.empty:
        roe = net_income.iloc[0] / equity.iloc[0]
        if roe > 0.15: profit_score += 20; reasons.append("Strong ROE (>15%).")
        elif roe < 0.05: profit_score -= 20; reasons.append("Weak ROE (<5%).")
    else:
        reasons.append("ROE data missing.")

    group_scores["Profitability"] = clamp_score(profit_score)

    # ========= GROUP 3: Momentum =========
    momentum_score = 50
    if len(hist) > 50:
        hist['MA20'] = hist['Close'].rolling(20).mean()
        hist['MA50'] = hist['Close'].rolling(50).mean()
        if hist['MA20'].iloc[-1] > hist['MA50'].iloc[-1]:
            momentum_score += 15; reasons.append("Positive momentum (MA20 > MA50).")
        else:
            momentum_score -= 10; reasons.append("Weak momentum (MA20 < MA50).")

    revenue = fin.get('Total Revenue', pd.Series(dtype=float)).dropna()
    if len(revenue) >= 2:
        rev_growth = (revenue.iloc[0] - revenue.iloc[1]) / abs(revenue.iloc[1])
        if rev_growth > 0.1: momentum_score += 15; reasons.append("Revenue growth >10% YoY.")
        elif rev_growth < 0: momentum_score -= 15; reasons.append("Revenue declining.")
    else:
        reasons.append("Insufficient revenue data.")

    group_scores["Momentum"] = clamp_score(momentum_score)

    # ========= GROUP 4: Volatility =========
    vol_score = 50
    beta = info.get('beta', 1)
    if beta < 1: vol_score += 20; reasons.append("Low volatility (Beta <1).")
    else: vol_score -= 10; reasons.append("High volatility (Beta >1).")
    group_scores["Low Volatility"] = clamp_score(vol_score)

    # ========= GROUP 5: Sentiment =========
    sentiment_score = 50
    sentiment = TextBlob(info.get("longBusinessSummary", "")).sentiment.polarity
    if sentiment > 0: sentiment_score += 15; reasons.append("Positive sentiment from market news.")
    else: sentiment_score -= 10; reasons.append("Negative market sentiment.")
    group_scores["Sentiment"] = clamp_score(sentiment_score)

    # ========= Horizon Impact =========
    if horizon:
        horizon = horizon.lower()
        if "short" in horizon:
            final_score = (group_scores["Momentum"] * 0.4 + group_scores["Sentiment"] * 0.2 +
                           group_scores["Valuation"] * 0.2 + group_scores["Profitability"] * 0.2)
            reasons.append("Short-term: Momentum & Sentiment weighted higher.")
        elif "mid" in horizon:
            final_score = np.mean(list(group_scores.values()))
            reasons.append("Mid-term: Balanced weighting across all factors.")
        else:
            final_score = (group_scores["Valuation"] * 0.3 + group_scores["Profitability"] * 0.3 +
                           group_scores["Low Volatility"] * 0.2 + group_scores["Momentum"] * 0.1 +
                           group_scores["Sentiment"] * 0.1)
            reasons.append("Long-term: Fundamentals weighted higher.")
    else:
        final_score = np.mean(list(group_scores.values()))

    # ========= Ownership Impact =========
    if own_stock and avg_price and quantity:
        current_price = info.get('currentPrice', None)
        if current_price:
            pl_ratio = (current_price - avg_price) / avg_price
            if pl_ratio < -0.2: final_score -= 5; reasons.append("Loss >20%: reduces score.")
            elif pl_ratio > 0.2: final_score += 5; reasons.append("Profit >20%: improves score.")

    final_score = clamp_score(final_score)

    decision = "BUY" if final_score >= 60 and not own_stock else \
               "HOLD" if final_score >= 50 else "SELL"

    return final_score, decision, reasons, group_scores


# ================== TECHNICAL INDICATORS FOR MODEL B ==================
def compute_RSI(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def compute_MACD(series, short=12, long=26, signal=9):
    ema_short = series.ewm(span=short, adjust=False).mean()
    ema_long = series.ewm(span=long, adjust=False).mean()
    macd = ema_short - ema_long
    signal_line = macd.ewm(span=signal, adjust=False).mean()
    return macd, signal_line


# ================== UPGRADED MODEL B ==================
def model_b_train_upgraded(hist):
    # --- Feature Engineering ---
    hist['MA5'] = hist['Close'].rolling(5).mean()
    hist['MA20'] = hist['Close'].rolling(20).mean()
    hist['MA50'] = hist['Close'].rolling(50).mean()
    hist['MA200'] = hist['Close'].rolling(200).mean()

    hist['RSI'] = compute_RSI(hist['Close'])
    hist['Returns'] = hist['Close'].pct_change()
    hist['Volatility'] = hist['Returns'].rolling(20).std()

    hist['BB_high'] = hist['MA20'] + 2 * hist['Volatility']
    hist['BB_low'] = hist['MA20'] - 2 * hist['Volatility']

    macd, signal = compute_MACD(hist['Close'])
    hist['MACD'] = macd
    hist['Signal'] = signal

    hist = hist.dropna()

    features = [
        'Close', 'Volume', 'MA5', 'MA20', 'MA50', 'MA200',
        'RSI', 'Returns', 'Volatility', 'BB_high', 'BB_low',
        'MACD', 'Signal'
    ]

    X = hist[features].values
    y = hist['Close'].values

    # --- Ensemble Model ---
    rf = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42)
    xgb = XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.05, random_state=42)

    model = VotingRegressor([('rf', rf), ('xgb', xgb)])
    model.fit(X, y)

    return model, features, hist


# ================= MAIN ANALYSIS =================
def run_analysis(stock, chart_range, own=False, avg_price=None, quantity=None, hold_plan=None, horizon=None):
    if "." not in stock:
        stock += ".NS"

    data = fetch_stock_data(stock, chart_range)
    if data is None:
        print(f"❌ Could not fetch data for {stock}.")
        return

    info, hist, fin, bs = data["info"], data["hist"], data["financials"], data["balance_sheet"]
    if hist.empty:
        print(f"❌ No price data for {stock}.")
        return

    # Model A
    score, decision, reasons, group_scores = compute_score_grouped_fixed(
        info, fin, bs, hist, own_stock=own, horizon=(hold_plan or horizon),
        avg_price=avg_price, quantity=quantity
    )

    # Model B (Upgraded)
    forecast_price, next_date = None, None
    if len(hist) > 250:  # Need at least 250 days
        model_b, features, hist_feat = model_b_train_upgraded(hist.copy())
        last_X = hist_feat[features].iloc[-1].values.reshape(1, -1)
        forecast_price = model_b.predict(last_X)[0]
        next_date = hist_feat.index[-1] + timedelta(days=1)

    # ========== OUTPUT ==========
    print(f"\n📊 {stock} Analysis Summary")
    print("=====================================")
    print(f"Final Score (Model ANALYSIS): {score}/100 → Decision: {decision}\n")

    # --- Tabular Summary ---
    print("📌 Factor Summary:")
    for factor, val in group_scores.items():
        if val < 40: status = "🔴 Poor"
        elif val < 70: status = "🟡 Average"
        else: status = "🟢 Good"
        print(f" - {factor:15}: {val:5.1f}/100  {status}")

    if forecast_price:
        print(f"\nPrice Forecast (Model PREDICTION ): ₹{forecast_price:.2f} expected on {next_date.date()}")

    print("=====================================")
    print("Reasons:")
    for r in reasons:
        print(f" - {r}")

    # --- Radar Chart ---
    categories = list(group_scores.keys())
    values = list(group_scores.values()) + [list(group_scores.values())[0]]
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.fill(angles, values, color='purple', alpha=0.25)
    ax.plot(angles, values, color='purple', linewidth=2)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_yticklabels([])
    ax.set_title(f"{stock} - Factor Radar Chart", fontsize=14)
    plt.show()

    # --- Price Chart ---
    hist['Close'].plot(figsize=(10, 4), label="Close Price")
    if 'MA20' in hist and 'MA50' in hist:
        hist['MA20'].plot(label='MA20')
        hist['MA50'].plot(label='MA50')
    plt.title(f"{stock} Price Trend (MA20/MA50)")
    plt.legend()
    plt.show()


# ================= WIDGET UI =================
ticker_input = widgets.Text(value='', description='Stock Ticker:')
time_period_dropdown = widgets.Dropdown(
    options=['6 Months', '1 Year', '2 Years', '5 Years'],
    value='2 Years', description='Time Period:'
)
own_stock_radio = widgets.RadioButtons(options=['Yes', 'No'], value='No', description='Own Stock?')
investment_horizon_dropdown = widgets.Dropdown(
    options=['<6 months', '6-12 months', '1-2 years', '2+ years'],
    value='1-2 years', description='Investment Horizon:'
)
avg_price_input = widgets.FloatText(value=0.0, description='Avg Buy Price:')
quantity_input = widgets.IntText(value=0, description='Quantity:')
hold_plan_dropdown = widgets.Dropdown(
    options=['Short-term', 'Mid-term', 'Long-term'],
    value='Mid-term', description='Hold Plan:'
)
run_button = widgets.Button(description='Run Analysis', button_style='primary', layout=widgets.Layout(width='300px'))
output = widgets.Output()


def toggle_inputs(change):
    if change['new'] == 'Yes':
        avg_price_input.layout.display = ''
        quantity_input.layout.display = ''
        hold_plan_dropdown.layout.display = ''
        investment_horizon_dropdown.layout.display = 'none'
    else:
        avg_price_input.layout.display = 'none'
        quantity_input.layout.display = 'none'
        hold_plan_dropdown.layout.display = 'none'
        investment_horizon_dropdown.layout.display = ''

own_stock_radio.observe(toggle_inputs, names='value')
toggle_inputs({'new': own_stock_radio.value})


def on_button_click(b):
    with output:
        clear_output()
        ticker = ticker_input.value.strip().upper()
        period_map = {'6 Months': '6mo', '1 Year': '1y', '2 Years': '2y', '5 Years': '5y'}
        chart_range = period_map[time_period_dropdown.value]
        if own_stock_radio.value == 'Yes':
            run_analysis(ticker, chart_range, own=True,
                         avg_price=avg_price_input.value, quantity=quantity_input.value,
                         hold_plan=hold_plan_dropdown.value)
        else:
            run_analysis(ticker, chart_range, own=False, horizon=investment_horizon_dropdown.value)

run_button.on_click(on_button_click)

ui = widgets.VBox([
    widgets.HTML("<h3>📈 Stock Analysis Parameters</h3>"),
    ticker_input,
    time_period_dropdown,
    own_stock_radio,
    investment_horizon_dropdown,
    avg_price_input,
    quantity_input,
    hold_plan_dropdown,
    run_button,
    output
])
display(ui)




C:\Users\debad\AppData\Local\Temp\ipykernel_24300\2989698663.py:13: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use('seaborn')


jupyter nbconvert --to script chart_model.ipynb

